In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
adata = sc.read_h5ad('../hpca_downstream/2026_final_object_cleaned.h5ad')

In [ ]:
adata

In [ ]:
pwd

In [ ]:
adata[adata.obs.Level_4_refined_desc.str.contains('High|Low|Mid')].obs.groupby(['leiden_3_subclustered_immune', 'Level_4_refined_desc']).size().unstack()

In [ ]:
adata.obs[adata.obs.columns[adata.obs.columns.str.contains('score')]]

# Alpha Beta Scores

In [ ]:
alpha_markers = ["GCG","ARX","MAFB","PCSK2"]
beta_markers = ["INS","MAFA","IAPP","PDX1"]

sc.tl.score_genes(adata, alpha_markers, score_name="alpha_identity_score")
sc.tl.score_genes(adata, beta_markers, score_name="beta_identity_score")

In [ ]:
adata[adata.obs.Level_4_refined_desc.str.contains('Beta|Alpha')].obs.groupby([
    'Level_4_refined_desc'])[['alpha_identity_score', 'beta_identity_score']].median()

In [ ]:
subset_adata = adata[adata.obs.Level_4_refined_desc.str.contains('Beta|Alpha')]

In [ ]:
subset_adata.obs["INS_expr"] = subset_adata[:, "INS"].X.toarray().flatten()
subset_adata.obs["MAFA_expr"] = subset_adata[:, "MAFA"].X.toarray().flatten()
subset_adata.obs["PDX1_expr"] = subset_adata[:, "PDX1"].X.toarray().flatten()
subset_adata.obs['IAPP_expr'] = subset_adata[:, "IAPP"].X.toarray().flatten()
subset_adata.obs["GCG_expr"] = subset_adata[:, "GCG"].X.toarray().flatten()
subset_adata.obs["ARX_expr"] = subset_adata[:, "ARX"].X.toarray().flatten()
subset_adata.obs["MAFB_expr"] = subset_adata[:, "MAFB"].X.toarray().flatten()
subset_adata.obs["PCSK2_expr"] = subset_adata[:, "PCSK2"].X.toarray().flatten()
subset_adata.obs['POU6F2_expr'] = subset_adata[:, "POU6F2"].X.toarray().flatten()
subset_adata.obs['IRX2_expr'] = subset_adata[:, "IRX2"].X.toarray().flatten()

In [ ]:
pd.set_option('display.max_columns', None)
subset_adata.obs.head()

In [ ]:
subset_adata[subset_adata.obs.Level_4_refined_desc.str.contains('Beta')].obs.groupby(['leiden_3_subclustered_immune'])['donor_entropy_leiden'].unique()

In [ ]:
# groupby leiden_3_subclustered_immune and Level_4_refined_desc and get mean of INS_expr, MAFA_expr, PDX1_expr, IAPP_expr, donor_entropy_leiden
beta = subset_adata[subset_adata.obs["Level_4_refined_desc"].str.contains("Beta", na=False)]
out = (
    beta.obs
    .groupby(["leiden_3_subclustered_immune", "Level_4_refined_desc", 'donor_entropy_leiden', 'dataset_entropy_leiden'], observed=True)
    .agg(
        Donor_Entropy=("donor_entropy_scaled", "mean"),
        Dataset_Entropy=("dataset_entropy_scaled", "mean"),
        INS_expr_mean=("INS_expr", "mean"),
        MAFA_expr_mean=("MAFA_expr", "mean"),
        IAPP_expr_mean=("IAPP_expr", "mean"),
        PDX1_expr_mean=("PDX1_expr", "mean"),
    )
    .reset_index()
)
out.sort_values(by='Level_4_refined_desc', inplace=True)

In [ ]:
out

In [ ]:
# distribution of INS_expr, MAFA_expr, IAPP_expr, PDX1_expr across cells in the beta subset
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.histplot(beta.obs['INS_expr'], bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of INS expression')
sns.histplot(beta.obs['MAFA_expr'], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of MAFA expression')
sns.histplot(beta.obs['IAPP_expr'], bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of IAPP expression')
sns.histplot(beta.obs['PDX1_expr'], bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution of PDX1 expression')
plt.tight_layout()
plt.show()

In [ ]:
# Define INS, MAFA, IAPP by cutting into High, Mid, Low based on quantiles per cell
def categorize_marker(series):
    s = series.copy()
    out = pd.Series(index=s.index, dtype="object")
    # 1. Off cells
    out[s == 0] = "No Expression"
    # 2. Nonzero cells
    nz = s[s > 0]
    if len(nz) == 0:
        out[:] = "No Expression"
    elif nz.nunique() == 1:
        out.loc[nz.index] = "High"
    else:
        out.loc[nz.index] = pd.qcut(
            nz,
            q=3,
            labels=["Low", "Mid", "High"]
        )

    return pd.Categorical(
        out,
        categories=["No Expression", "Low", "Mid", "High"],
        ordered=True
    )

for gene in ["INS_expr", "MAFA_expr", "PDX1_expr", "IAPP_expr"]:
    beta.obs[gene.replace("_expr", "_category")] = categorize_marker(
        beta.obs[gene]
    )

In [ ]:
def category_proportions(x):
    counts = x.value_counts(normalize=True)
    return {
        "NoExp_frac": counts.get("No Expression", 0),
        "Low_frac": counts.get("Low", 0),
        "Mid_frac": counts.get("Mid", 0),
        "High_frac": counts.get("High", 0),
    }

out = (
    beta.obs
    .groupby(
        ["leiden_3_subclustered_immune",
         "Level_4_refined_desc",
         "donor_entropy_leiden",
         "dataset_entropy_leiden"],
        observed=True
    )
    .apply(lambda df: pd.Series({
        "Donor_Entropy": df["donor_entropy_scaled"].mean(),
        "Dataset_Entropy": df["dataset_entropy_scaled"].mean(),

        "INS_expr_mean": df["INS_expr"].mean(),
        **{f"INS_{k}": v for k, v in category_proportions(df["INS_category"]).items()},

        "MAFA_expr_mean": df["MAFA_expr"].mean(),
        **{f"MAFA_{k}": v for k, v in category_proportions(df["MAFA_category"]).items()},

        "IAPP_expr_mean": df["IAPP_expr"].mean(),
        **{f"IAPP_{k}": v for k, v in category_proportions(df["IAPP_category"]).items()},

        "PDX1_expr_mean": df["PDX1_expr"].mean(),
    }))
    .reset_index()
).sort_values(by='Level_4_refined_desc')

In [ ]:
out

### change of cluster names
- 35 Alpha-Beta-Delta Cell: INS_low frac 0.59, INS_mid 0.35, MAFA No_expression 0.50, no clear IAPP dominance 
- 58 Alpha-Beta-Delta Cell: INS_low frac 0.62, MAFA_high 0.33, IAPP_high 0.36 
- 41 Beta Cell [INS_High, MAFA_IAPP_Mid]: Beta Cell [INS_High] INS_high frac 0.61, MAFA No_expression 0.57 so no need to report, IAPP expression dominated by IAPP_low with 0.32
- 67 Beta Cell [INS_High, MAFA_IAPP_Mid]: Beta Cell [INS_High, IAPP_Low] INS_high frac 0.66, MAFA No_expression 0.58 so no need to report, IAPP_low dominant with 0.49
- 76 Beta Cell [INS_High, MAFA_IAPP_Mid]: Beta Cell [INS_Mid, ] INS_mid frac 0.50, MAFA_high enriched 0.29 (not dominated), IAPP broadly distributed but no clear suppression
- 49 Beta Cell [INS_High_MAFA_Low]: Beta Cell [INS_High] INS_high frac 0.67, MAFA No_expression dominant 0.92, IAPP mostly low/mid
- 8 Beta Cell [INS_Low]: Beta Cell [INS_Low] INS_low frac 0.76, MAFA No_expression 0.72, IAPP largely No_expression/Low
- 46 Beta Cell [INS_Low]: Beta Cell [INS_Low, IAPP_Low] INS_low frac 0.96, MAFA No_expression 0.60, IAPP_low dominant 0.34
- 4 Beta Cell [INS_MAFA_High]: Beta Cell [INS_Mid, IAPP_High] IAPP_high dominant 0.51, MAFA_high enriched 0.20, INS mostly mid/low (identity-driven state)
- 77 Beta Cell [INS_MAFA_High]: Beta Cell [INS_Mid, IAPP_High] INS_mid dominant 0.78, IAPP_high 0.36, MAFA balanced, donor-fragmented version of cluster 4
- 80 Beta Cell [Secretion-High: INS_High, MAFA_Low]: Beta Cell [INS_Low] INS_low dominant 0.96, MAFA No_expression 0.80, very low entropy so likely donor-driven
- 42 Beta Cell [Stress: INS_High, MAFA_Low]: Beta Cell [INS_High, IAPP_Low] INS_high frac 0.58, MAFA No_expression 0.73, IAPP_low dominant 0.37

#### Report MAFA or IAPP only if:
	1.	No_expression is NOT dominant
	2.	One of Low/Mid/High ≥ 0.4

In [ ]:
cluster_rename_map = {
    '35': "Alpha-Beta-Delta Cell",
    '58': "Alpha-Beta-Delta Cell",
    '41': "Beta Cell [INS_High]",
    '49': "Beta Cell [INS_High]",
    '42': "Beta Cell [INS_High]",
    '67': "Beta Cell [INS_High, IAPP_Low]", #IAPP_low = 0.49
    '76': "Beta Cell [INS_Mid]",
    '77': "Beta Cell [INS_Mid]",
    '4':  "Beta Cell [INS_Mid, IAPP_High]", #IAPP_high = 0.51 
    '8':  "Beta Cell [INS_Low]",
    '46': "Beta Cell [INS_Low]",
    '80': "Beta Cell [INS_Low]"}

In [ ]:
beta.obs['Level_4_redone'] = beta.obs['leiden_3_subclustered_immune'].map(cluster_rename_map)

In [ ]:
sc.pl.umap(beta, color=['leiden_3_subclustered_immune', 'Level_4_refined_desc', 'Level_4_redone'], wspace=0.4, ncols=1)

# Redo For Alpha cells

In [ ]:
alpha = subset_adata[subset_adata.obs["Level_4_refined_desc"].str.contains("Alpha", na=False)]

In [ ]:
alpha[alpha.obs.Level_4_refined_desc.str.contains('Alpha')].obs.groupby(['Level_4_refined_desc'])[[
    'GCG_expr', 'POU6F2_expr', 'IRX2_expr', 'MAFB_expr']].median()

In [ ]:
out = (
    alpha.obs
    .groupby(["leiden_3_subclustered_immune", "Level_4_refined_desc", 'donor_entropy_leiden', 'dataset_entropy_leiden'], observed=True)
    .agg(
        Donor_Entropy=("donor_entropy_scaled", "mean"),
        Dataset_Entropy=("dataset_entropy_scaled", "mean"),
        GCG_expr_mean=("GCG_expr", "mean"),
        POU6F2_expr_mean=("POU6F2_expr", "mean"),
        IRX2_expr_mean=("IRX2_expr", "mean"),
        MAFB_expr_mean=("MAFB_expr", "mean"),
        INS_expr_mean=("INS_expr", "mean"),
    )
    .reset_index()
)
out.sort_values(by='Level_4_refined_desc', inplace=True)

In [ ]:
out

In [ ]:
# distribution of INS_expr, MAFA_expr, IAPP_expr, PDX1_expr across cells in the beta subset
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.histplot(alpha.obs['GCG_expr'], bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of GCG expression')
sns.histplot(alpha.obs['MAFB_expr'], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Distribution of MAFB expression')
sns.histplot(alpha.obs['POU6F2_expr'], bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Distribution of POU6F2 expression')
sns.histplot(alpha.obs['IRX2_expr'], bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Distribution of IRX2 expression')
plt.tight_layout()
plt.show()

In [ ]:
for gene in ["GCG_expr", "MAFB_expr", "POU6F2_expr", "IRX2_expr"]:
    alpha.obs[gene.replace("_expr", "_category")] = categorize_marker(
        alpha.obs[gene]
    )

In [ ]:
out = (
    alpha.obs
    .groupby(
        ["leiden_3_subclustered_immune",
         "Level_4_refined_desc",
         "donor_entropy_leiden",
         "dataset_entropy_leiden"],
        observed=True
    )
    .apply(lambda df: pd.Series({
        "Donor_Entropy": df["donor_entropy_scaled"].mean(),
        "Dataset_Entropy": df["dataset_entropy_scaled"].mean(),

        "GCG_expr_mean": df["GCG_expr"].mean(),
        **{f"GCG_{k}": v for k, v in category_proportions(df["GCG_category"]).items()},

        "MAFB_expr_mean": df["MAFB_expr"].mean(),
        **{f"MAFB_{k}": v for k, v in category_proportions(df["MAFB_category"]).items()},

        "POU6F2_expr_mean": df["POU6F2_expr"].mean(),
        **{f"POU6F2_{k}": v for k, v in category_proportions(df["POU6F2_category"]).items()},

        "IRX2_expr_mean": df["IRX2_expr"].mean(),
        **{f"IRX2_{k}": v for k, v in category_proportions(df["IRX2_category"]).items()},
    }))
    .reset_index()
).sort_values(by='Level_4_refined_desc')

In [ ]:
out

In [ ]:
alpha_cluster_rename_map = {
    # --- Core alpha states ---
    '29': "Alpha Cell [GCG_Low]",
    '56': "Alpha Cell [GCG_Low]",
    '16': "Alpha Cell [GCG_Low]",

    '72': "Alpha Cell [GCG_Mid]",
    '43': "Alpha Cell [GCG_Mid]",
    '20': "Alpha Cell [GCG_Mid]",

    '52': "Alpha Cell [GCG_High]",
    '34': "Alpha Cell [GCG_High]",
    '54': "Alpha Cell [GCG_High]",

    '37': "Alpha Cell [GCG_Low, POU6F2_High]",
    '59': "Alpha Cell [GCG_High, IRX2_High]",

    '35': "Alpha-Beta-Delta Cell",
    '58': "Alpha-Beta-Delta Cell",
}

In [ ]:
alpha.obs["Level_4_redone"] = (
    alpha.obs["leiden_3_subclustered_immune"]
    .map(alpha_cluster_rename_map)
)

In [ ]:
sc.pl.umap(alpha, color=['leiden_3_subclustered_immune', 'Level_4_refined_desc', 'Level_4_redone', ], wspace=0.4, ncols=1)

In [ ]:
# plot in one fig
fig, ax = plt.subplots(1, 3, figsize=(30, 6))
sc.pl.umap(alpha, color=['Level_4_redone', ], groups=['Alpha Cell [GCG_Low]'], ax=ax[0], show=False)
sc.pl.umap(alpha, color=['annotation_sterr', ], groups=['Alpha Cell [GCG_Low]'], ax=ax[0], show=False)
sc.pl.umap(alpha, color=['leiden_3_subclustered_immune',], groups=['16', '29', '56'], ncols=1, ax=ax[1], show=False)
sc.pl.umap(alpha, color=['Dataset', ],ncols=1, ax=ax[2], show=False)
plt.tight_layout()
plt.show()

In [ ]:
alpha[alpha.obs.Level_4_redone == 'Alpha Cell [GCG_Low]'].obs.groupby(['leiden_3_subclustered_immune', 'Dataset']).size().unstack().T

In [ ]:
gcg_low = alpha[alpha.obs.Level_4_redone == 'Alpha Cell [GCG_Low]']
sc.tl.rank_genes_groups(gcg_low, groupby='leiden_3_subclustered_immune', method='wilcoxon')
pd.DataFrame(gcg_low.uns['rank_genes_groups']['names']).head(200).to_csv('gcg_low_markers.csv', index=False)

In [ ]:
# add Level_4_redone from alpha and beta to the original adata object with loc
adata.obs['Level_4_redone'] = adata.obs.Level_4_refined_desc.copy().astype(str)
adata.obs.loc[beta.obs.index, 'Level_4_redone'] = beta.obs['Level_4_redone']
adata.obs.loc[alpha.obs.index, 'Level_4_redone'] = alpha.obs['Level_4_redone']
adata.obs.Level_4_redone.value_counts().sort_index()

In [ ]:
sc.pl.umap(adata, color=['Level_4_refined_desc', 'Level_4_redone'], wspace=0.4, ncols=1)

In [ ]:
beta.obs.groupby(['Level_4_redone', 'Level_4_refined_desc']).size().unstack()

In [ ]:
adata.obs.Level_4_redone.value_counts().sort_index()

In [ ]:
# # Start from a copy
# adata.obs["cell_states_plus_L4"] = adata.obs["cell_states"].copy().astype(str)
# mask = adata.obs["cell_states_plus_L4"].str.contains("Baseline")
# adata.obs.loc[mask, "cell_states_plus_L4"] = adata.obs.loc[mask, "Level_4_redone"]

In [ ]:
# adata[adata.obs.cell_states_plus_L4.str.contains('Beta')].obs['cell_states_plus_L4'].value_counts().sort_index()

In [ ]:
# sc.pl.umap(adata[adata.obs.cell_states_plus_L4.str.contains('Beta')], color=['cell_states_plus_L4'], wspace=0.4, ncols=1)

In [ ]:
# sc.pl.umap(adata[adata.obs.cell_states_plus_L4.str.contains('Alpha')], color=['cell_states_plus_L4'], wspace=0.4, ncols=1)

In [ ]:
adata.write('../hpca_downstream/2026_final_object_cell_states.h5ad')

In [ ]:
adata

In [ ]:
adata.obs.Level_4_redone.value_counts().sort_index()